# 多层感知机训练示例（4-128-64-2）

本笔记本演示如何训练一个多层感知机（MLP），用于根据当前工况预测两步目标油面高度。模型与数据的关键要求如下：

- **输入特征（4 列）**：`L0`、`Deg0`、`matched_L`、`matched_deg`
- **输出标签（2 列）**：`L1`、`L2`
- **网络结构**：4 → 128 → 64 → 2，隐藏层使用 `ReLU`，输出层使用 `0.5 * Sigmoid` 以保证预测范围在 0–0.5 之间
- **归一化**：训练前对所有输入特征进行标准化处理（均值为 0，方差为 1）
- **损失函数**：在均方误差的基础上，加入对动作幅度的惩罚项，符合“尽量小动作”的需求

请准备包含上述列名的 CSV 文件：

- 训练集：`TRAIN_CSV_PATH`
- 测试集：`TEST_CSV_PATH`

每列含义如下：

| 列名 | 说明 |
| --- | --- |
| `L0` | 当前油面高度 |
| `Deg0` | 当前相位角 |
| `matched_L` | 匹配后的负载高度 |
| `matched_deg` | 匹配后的负载相位 |
| `L1` | 第一步目标油面高度 |
| `L2` | 第二步目标油面高度 |

后续单元会说明如何加载数据、训练模型、保存训练曲线以及在测试集上生成预测预览。

In [ ]:
# 导入所需库
import os
from copy import deepcopy
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from plotting_utils import ensure_plot_dir, save_loss_vs_epoch, save_training_curves

# 为可复现性设定随机种子
torch.manual_seed(42)

# 自动选择计算设备
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备：{DEVICE}")

In [ ]:
# TODO: 将以下路径替换为您本地的训练/测试 CSV 文件路径
TRAIN_CSV_PATH = "./data/train.csv"
TEST_CSV_PATH = "./data/test.csv"

for path in [TRAIN_CSV_PATH, TEST_CSV_PATH]:
    if not Path(path).exists():
        print(f"警告：未找到 {path}，请更新为实际数据文件路径。")

PLOT_DIR = ensure_plot_dir("./plots")
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
class OilLevelDataset(Dataset):
    """加载油面调节数据集并对输入特征做标准化。"""

    feature_columns = ["L0", "Deg0", "matched_L", "matched_deg"]
    target_columns = ["L1", "L2"]

    def __init__(self, csv_path: str | Path, *, stats: Tuple[np.ndarray, np.ndarray] | None = None):
        df = pd.read_csv(csv_path)
        missing = [col for col in self.feature_columns + self.target_columns if col not in df.columns]
        if missing:
            raise ValueError(f"数据缺少必要列：{missing}")

        features = df[self.feature_columns].astype(np.float32)
        targets = df[self.target_columns].astype(np.float32)

        if stats is None:
            mean = features.mean()
            std = features.std().replace(0.0, 1.0)
        else:
            mean_arr, std_arr = stats
            mean = pd.Series(mean_arr, index=self.feature_columns)
            std = pd.Series(std_arr, index=self.feature_columns).replace(0.0, 1.0)

        normalized = (features - mean) / std

        self.features = torch.from_numpy(normalized.to_numpy(dtype=np.float32))
        self.targets = torch.from_numpy(targets.to_numpy(dtype=np.float32))
        self.baselines = torch.from_numpy(df["L0"].astype(np.float32).to_numpy().reshape(-1, 1))
        self.raw_dataframe = df.reset_index(drop=True)

        self.feature_mean = mean.to_numpy(dtype=np.float32)
        self.feature_std = std.to_numpy(dtype=np.float32)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int):
        return self.features[idx], self.targets[idx], self.baselines[idx]

    def get_feature_stats(self) -> Tuple[np.ndarray, np.ndarray]:
        return self.feature_mean, self.feature_std

In [ ]:
class OilLevelMLP(nn.Module):
    """4-128-64-2 的多层感知机，隐藏层 ReLU，输出层 0.5*Sigmoid。"""

    def __init__(self, output_scale: float = 0.5):
        super().__init__()
        self.output_scale = output_scale
        self.backbone = nn.Sequential(
            nn.Linear(4, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.backbone(x)
        return self.output_scale * self.sigmoid(logits)

In [ ]:
class MinimalAdjustmentLoss(nn.Module):
    """loss = MSE + alpha * 平均动作幅度，符合“尽量小动作”要求。"""

    def __init__(self, alpha: float = 0.1):
        super().__init__()
        self.alpha = alpha

    def forward(self, predictions: torch.Tensor, targets: torch.Tensor, baseline_levels: torch.Tensor) -> torch.Tensor:
        mse = torch.mean((predictions - targets) ** 2)

        baseline_levels = baseline_levels.view(-1, 1)
        sequence = torch.cat([baseline_levels, predictions], dim=1)
        deltas = torch.abs(sequence[:, 1:] - sequence[:, :-1])
        avg_adjustment = torch.mean(torch.sum(deltas, dim=1))

        return mse + self.alpha * avg_adjustment

In [ ]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: MinimalAdjustmentLoss,
) -> float:
    model.train()
    total_loss = 0.0
    total_samples = 0

    for inputs, targets, baselines in loader:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)
        baselines = baselines.to(DEVICE)

        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, targets, baselines)
        loss.backward()
        optimizer.step()

        batch_size = inputs.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / max(total_samples, 1)


def evaluate_epoch(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: MinimalAdjustmentLoss,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_abs_error = 0.0
    total_samples = 0
    num_outputs = loader.dataset.targets.shape[1]

    with torch.no_grad():
        for inputs, targets, baselines in loader:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            baselines = baselines.to(DEVICE)

            predictions = model(inputs)
            loss = loss_fn(predictions, targets, baselines)

            batch_size = inputs.size(0)
            total_loss += loss.item() * batch_size
            total_abs_error += torch.sum(torch.abs(predictions - targets)).item()
            total_samples += batch_size

    mean_loss = total_loss / max(total_samples, 1)
    mae = total_abs_error / max(total_samples * num_outputs, 1)
    return mean_loss, mae

In [ ]:
# 训练模型
batch_size = 64
num_epochs = 200
learning_rate = 1e-3
alpha = 0.1

train_dataset = OilLevelDataset(TRAIN_CSV_PATH)
feature_stats = train_dataset.get_feature_stats()
test_dataset = OilLevelDataset(TEST_CSV_PATH, stats=feature_stats)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

model = OilLevelMLP().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = MinimalAdjustmentLoss(alpha=alpha)

train_history: list[float] = []
test_history: list[dict[str, float]] = []
metrics_records: list[dict[str, float]] = []
best_state = None
best_test_loss = float('inf')

for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
    test_loss, test_mae = evaluate_epoch(model, test_loader, loss_fn)

    train_history.append(train_loss)
    test_history.append({'loss': test_loss, 'mae': test_mae})
    metrics_records.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'test_loss': test_loss,
        'test_mae': test_mae,
    })

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_state = deepcopy(model.state_dict())

    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{num_epochs} - Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f} | Test MAE: {test_mae:.6f}")

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"
已恢复测试集损失最优的模型参数（Test Loss = {best_test_loss:.6f}）。")
else:
    print("
未找到更优模型状态，使用最终训练权重。")

metrics_df = pd.DataFrame(metrics_records)
print("训练指标预览：")
display(metrics_df.head())

In [ ]:
# 保存训练指标并绘制曲线
metrics_csv_path = PLOT_DIR / 'training_metrics.csv'
metrics_df.to_csv(metrics_csv_path, index=False)
print(f"训练指标已保存到 {metrics_csv_path}。")

epoch_indices = metrics_df['epoch'].tolist()
train_losses = metrics_df['train_loss'].tolist()
test_losses = metrics_df['test_loss'].tolist()
test_maes = metrics_df['test_mae'].tolist()

plot_paths = save_training_curves(epoch_indices, train_losses, test_losses, test_maes, PLOT_DIR)
if plot_paths.get('loss_curves'):
    print(f"损失曲线图已保存到 {plot_paths['loss_curves']}")
else:
    print('未检测到 Pillow 库，已跳过损失曲线绘制。')

if plot_paths.get('test_mae'):
    print(f"MAE 曲线图已保存到 {plot_paths['test_mae']}")

In [ ]:
@torch.no_grad()
def predict_to_dataframe(model: nn.Module, dataset: OilLevelDataset) -> pd.DataFrame:
    model.eval()
    loader = DataLoader(dataset, batch_size=128, shuffle=False)
    predictions: list[torch.Tensor] = []

    for inputs, _, _ in loader:
        preds = model(inputs.to(DEVICE)).cpu()
        predictions.append(preds)

    if not predictions:
        return dataset.raw_dataframe.copy()

    concatenated = torch.cat(predictions, dim=0).numpy()
    df = dataset.raw_dataframe.copy()
    df['Pred_L1'] = concatenated[:, 0]
    df['Pred_L2'] = concatenated[:, 1]
    return df

In [ ]:
# 生成测试集预测并保存对比结果
comparison_df = predict_to_dataframe(model, test_dataset)
preview_columns = ['L1', 'L2', 'Pred_L1', 'Pred_L2']

print('测试集实际值与预测值预览：')
display(comparison_df[preview_columns].head())

comparison_csv = RESULTS_DIR / 'test_predictions_vs_actuals.csv'
comparison_df.to_csv(comparison_csv, index=False)
print(f"完整对比结果已保存到 {comparison_csv}。")

actual = comparison_df[['L1', 'L2']].to_numpy(dtype=np.float32)
predicted = comparison_df[['Pred_L1', 'Pred_L2']].to_numpy(dtype=np.float32)
errors = predicted - actual

mse = float(np.mean(errors ** 2))
rmse = float(np.sqrt(mse))
mae = float(np.mean(np.abs(errors)))
ss_res = float(np.sum(errors ** 2))
ss_tot = float(np.sum((actual - actual.mean(axis=0)) ** 2))
r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 1.0

print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"R^2:  {r2:.6f}")

In [ ]:
# 仅针对损失函数重新绘图（可选）
loss_plot_path = save_loss_vs_epoch(epoch_indices, train_losses, test_losses, PLOT_DIR)
if loss_plot_path:
    print(f"损失函数曲线已保存到 {loss_plot_path}")
else:
    print('未检测到 Pillow 库，跳过损失函数曲线绘制。')

In [ ]:
# 绘制损失函数曲线模块
final_loss_plot = save_loss_vs_epoch(epoch_indices, train_losses, test_losses, PLOT_DIR, output_name='loss_curves_final.png')
if final_loss_plot:
    print(f"最终损失函数曲线已保存到 {final_loss_plot}")
else:
    print('未检测到 Pillow 库或输入数据为空，跳过损失函数曲线绘制。')
